# Full fine-tuning Gemma-3-1B on a free T4: paraphrase generation

**Colab, T4 GPU.** `Runtime` → `Change runtime type` → **T4 GPU**, then `Run all`.
Edit `HF_USERNAME` in section 12. Roughly 70–90 minutes.

**Full fine-tuning** — every weight updated, no adapters — of
[`google/gemma-3-1b-it`](https://huggingface.co/google/gemma-3-1b-it) with
[Unsloth](https://github.com/unslothai/unsloth).

### Gemma 3 cannot be trained in float16, and that decides everything here

Unsloth's own analysis: **Gemma 3 activations exceed float16's maximum of 65,504**, so
gradients and activations become infinity on T4, RTX 20-series and V100 — any card with
float16 tensor cores but no bfloat16. It is not a bug in the training script; it is the
model's dynamic range. Unsloth's fix is to **default every layer to float32** on those
devices.

So this run trains in **pure fp32**, not mixed precision. The memory arithmetic:

| | GB |
|---|---|
| weights, fp32 | 4.0 |
| gradients, fp32 (non-embedding) | 2.8 |
| AdamW states, 8-bit | 1.4 |
| activations + CUDA context | ~1.6 |
| **total** | **~9.8 of 15.0** |

That fits because **the embedding layer is frozen**. Gemma 3's vocabulary is 262,144
tokens — `embed_tokens` is 302M parameters, **30% of this entire model**, with `lm_head`
tied to it. Paraphrasing introduces no new vocabulary, so training it buys almost nothing
and costs gradients plus optimizer state for nearly a third of the network.

### Three more Gemma-specific traps

| Trap | What happens | Fix |
|---|---|---|
| `FastLanguageModel` | Gemma 3 needs the multimodal-aware loader | `FastModel` |
| Chat template adds `<bos>` itself | two BOS tokens, degraded training | `removeprefix("<bos>")` |
| No `system` role | template error or silently dropped instruction | instruction folded into the user turn |

### An honest note up front

Unsloth's guidance is that full fine-tuning is *usually unnecessary* and LoRA matches it
when done properly. This notebook does it because it was asked for, and because the fp32
memory wall is worth seeing. The card records that caveat rather than implying full
fine-tuning is the superior choice.

In [1]:
import subprocess
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode().strip()
    print(out)
    if "T4" not in out:
        print("Not a T4 — memory settings below are tuned for 16 GB Turing.")
except Exception:
    print("No GPU — Runtime > Change runtime type > T4 GPU")

Tesla T4, 15360 MiB


## 1. Install

In [2]:
!pip install -q unsloth
!pip install -q --no-deps --upgrade git+https://github.com/unslothai/unsloth.git
!pip install -q sentence-transformers sacrebleu evaluate rouge_score matplotlib
print("installed")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
installed


## 2. Configuration

In [3]:
import os, gc, re, ast, json, time, random, inspect
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from unsloth import FastModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset, Dataset
from trl import SFTTrainer

BASE_MODEL  = "unsloth/gemma-3-1b-it"        # ungated mirror of google/gemma-3-1b-it
UPSTREAM    = "google/gemma-3-1b-it"
MODEL_NAME  = "gemma-3-1b-paraphrase"

MAX_SEQ    = 256       # sentences, not documents
BATCH      = 4
ACCUM      = 4         # effective 16
MAX_STEPS  = 2500       # ~14,400 examples (fp32 is slower than fp16)
LR         = 1e-5      # full FT, not LoRA — see section 8
SEED       = 3407
FREEZE_EMBED = True    # 21% of the model; the memory wall depends on this

EVAL_N   = 300
MAX_NEW  = 72
TRAIN_POOL = 40000

SAVE_DIR = f"/content/{MODEL_NAME}"
ASSETS   = f"{SAVE_DIR}/assets"
os.makedirs(ASSETS, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GPU  = torch.cuda.get_device_name(0)
BF16 = is_bfloat16_supported()
print(f"{GPU} | bfloat16: {BF16}")
print("Gemma 3 overflows float16, so training runs in "
      f"{'bfloat16' if BF16 else 'pure float32'} on this device.")

plt.rcParams.update({"figure.dpi":130,"savefig.dpi":130,"savefig.bbox":"tight",
                     "font.size":10,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":.25,"figure.facecolor":"white"})
C_COPY, C_BASE, C_TUNED = "#cbd5e1", "#94a3b8", "#1a73e8"

def vram(): return torch.cuda.max_memory_allocated()/1e9
def clear(): gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Tesla T4 | bfloat16: False
Gemma 3 overflows float16, so training runs in pure float32 on this device.


## 3. Dataset

`humarin/chatgpt-paraphrases` pairs a source sentence with several rewrites. A fallback
to PAWS is in place in case the primary dataset moves or changes shape — the same kind of
breakage that has bitten every notebook in this series.

In [4]:
def load_pairs():
    try:
        d = load_dataset("humarin/chatgpt-paraphrases", split="train")
        cols = d.column_names
        assert "text" in cols and "paraphrases" in cols, f"unexpected columns {cols}"
        d = d.shuffle(seed=SEED).select(range(min(TRAIN_POOL * 2, len(d))))
        src, tgt = [], []
        for r in d:
            try:
                ps = r["paraphrases"]
                ps = ast.literal_eval(ps) if isinstance(ps, str) else ps
                if ps:
                    src.append(r["text"]); tgt.append(ps[0])
            except Exception:
                continue
        return src, tgt, "humarin/chatgpt-paraphrases"
    except Exception as e:
        print("primary dataset failed:", str(e)[:140])
        d = load_dataset("google-research-datasets/paws", "labeled_final", split="train")
        d = d.filter(lambda r: r["label"] == 1).shuffle(seed=SEED)
        return list(d["sentence1"]), list(d["sentence2"]), "google-research-datasets/paws"

SRC, TGT, DATASET = load_pairs()
print(f"{DATASET}: {len(SRC):,} pairs")

def ok(s, t):
    if not s or not t: return False
    s, t = s.strip(), t.strip()
    if s.lower() == t.lower(): return False                 # not a paraphrase
    if not (3 <= len(s.split()) <= 40): return False
    if not (3 <= len(t.split()) <= 48): return False
    return 0.5 <= len(t.split())/len(s.split()) <= 2.0      # drop truncations and ramblings

pairs = [(s.strip(), t.strip()) for s, t in zip(SRC, TGT) if ok(s, t)]
seen, clean = set(), []
for s, t in pairs:                                           # dedupe on source
    k = s.lower()
    if k not in seen:
        seen.add(k); clean.append((s, t))
print(f"after filtering and dedupe: {len(clean):,}")

random.shuffle(clean)
EVAL_PAIRS  = clean[:EVAL_N]
TRAIN_PAIRS = clean[EVAL_N:EVAL_N + TRAIN_POOL]
print(f"train {len(TRAIN_PAIRS):,} | eval {len(EVAL_PAIRS):,}")
print(f"\nexample\n  source: {TRAIN_PAIRS[0][0]}\n  target: {TRAIN_PAIRS[0][1]}")

humarin/chatgpt-paraphrases: 80,000 pairs
after filtering and dedupe: 77,099
train 40,000 | eval 300

example
  source: Jio Sim Problem: I used the IMEI method to get barcode on my samsung phone.Wil it activate even so?
  target: Can the barcode obtained through the IMEI method on my Samsung phone activate my Jio SIM despite any potential issues?


## 4. Model, in full fine-tuning mode

In [5]:
clear()
t0 = time.time()
model, tokenizer = FastModel.from_pretrained(     # FastModel, not FastLanguageModel
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ,
    dtype           = None,      # Unsloth picks fp32 for Gemma 3 on float16 devices
    load_in_4bit    = False,
    load_in_8bit    = False,
    full_finetuning = True,
)
LOAD_S = time.time()-t0
VRAM_LOAD = vram()
N_PARAMS = sum(p.numel() for p in model.parameters())
WDTYPE = next(model.parameters()).dtype
print(f"loaded in {LOAD_S:.0f}s | {N_PARAMS/1e9:.2f}B params | {VRAM_LOAD:.2f} GB")
print(f"weight dtype: {WDTYPE}")
if not BF16:
    assert WDTYPE == torch.float32, (
        f"Expected float32 on a float16 device for Gemma 3, got {WDTYPE}. "
        "Update unsloth — float16 will produce infinite gradients.")

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

# Gemma 3's template has no system role, so the instruction lives in the user turn.
INSTRUCTION = ("Rewrite the sentence below. Keep the meaning exactly the same, change the "
               "wording and structure, and return only the rewritten sentence.\n\n{src}")

def to_chat(src, tgt=None):
    m = [{"role": "user", "content": INSTRUCTION.format(src=src)}]
    if tgt is not None:
        m.append({"role": "model", "content": tgt})   # Gemma calls it "model", not "assistant"
    return m

def render(src, tgt=None, generation=False):
    """Gemma's template prepends <bos> itself; a second one degrades training."""
    t = tokenizer.apply_chat_template(to_chat(src, tgt), tokenize=False,
                                      add_generation_prompt=generation)
    return t.removeprefix(tokenizer.bos_token or "<bos>")

print("\n--- rendered training example ---")
print(render("The cat sat on the mat.", "A cat was sitting on the rug."))
assert not render("x", "y").startswith("<bos>"), "double BOS not stripped"

==((====))==  Unsloth 2026.10.5: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Not planning a device map; full finetuning does not use the quantized planner. Using `sequential`.
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


loaded in 43s | 1.00B params | 4.02 GB
weight dtype: torch.float32

--- rendered training example ---
<start_of_turn>user
Rewrite the sentence below. Keep the meaning exactly the same, change the wording and structure, and return only the rewritten sentence.

The cat sat on the mat.<end_of_turn>
<start_of_turn>model
A cat was sitting on the rug.<end_of_turn>



## 5. Freeze the embeddings

Gemma 3's vocabulary is **262,144 tokens** — far larger than Llama's 128k — so
`embed_tokens` is 262,144 × 1,152 = **302M parameters, about 30% of this entire model**,
with `lm_head` tied to it. Paraphrasing introduces no new vocabulary, so training it buys
almost nothing while costing gradients plus optimizer state for nearly a third of the
network.

In fp32 that is the difference between fitting on a T4 and not.

In [6]:
FROZEN = 0
if FREEZE_EMBED:
    for name, p in model.named_parameters():
        if "embed_tokens" in name or "lm_head" in name:
            p.requires_grad = False
            FROZEN += p.numel()

TRAINABLE = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total     {N_PARAMS:,}")
print(f"frozen    {FROZEN:,} ({100*FROZEN/N_PARAMS:.1f}%)")
print(f"trainable {TRAINABLE:,} ({100*TRAINABLE/N_PARAMS:.1f}%)")
assert TRAINABLE > 0.5 * N_PARAMS, "This is no longer full fine-tuning."

total     999,885,952
frozen    301,989,888 (30.2%)
trainable 697,896,064 (69.8%)


## 6. Evaluation: paraphrasing needs two numbers, not one

A single score cannot grade a paraphraser, because the two things it must do pull against
each other:

- **preserve the meaning** — maximised by copying the input verbatim
- **change the wording** — maximised by writing something unrelated

So a model that returns the input unchanged scores a perfect 1.0 on semantic similarity
and is useless. The **copy baseline** below exists to make that visible, and it is the
baseline most paraphrase model cards quietly omit.

`iBLEU` is the standard combined measure: reward overlap with a human reference, penalise
overlap with the input.

In [7]:
import sacrebleu
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
rouge = __import__("evaluate").load("rouge")

EV_SRC = [s for s, _ in EVAL_PAIRS]
EV_REF = [t for _, t in EVAL_PAIRS]

def cosine(a, b):
    ea = embedder.encode(a, convert_to_numpy=True, normalize_embeddings=True,
                         batch_size=64, show_progress_bar=False)
    eb = embedder.encode(b, convert_to_numpy=True, normalize_embeddings=True,
                         batch_size=64, show_progress_bar=False)
    return float(np.mean(np.sum(ea*eb, axis=1)))

def sent_bleu(hyps, refs):
    return sacrebleu.corpus_bleu(hyps, [refs]).score

def evaluate_system(outs, name):
    sim   = cosine(EV_SRC, outs)                     # meaning kept?
    b_in  = sent_bleu(outs, EV_SRC)                  # overlap with the input
    b_ref = sent_bleu(outs, EV_REF)                  # overlap with a human paraphrase
    ibleu = 0.8*b_ref - 0.2*b_in                     # standard paraphrase measure
    rl    = rouge.compute(predictions=outs, references=EV_REF,
                          use_stemmer=True)["rougeL"]*100
    copy  = 100*float(np.mean([o.strip().lower() == s.strip().lower()
                               for o, s in zip(outs, EV_SRC)]))
    lenr  = float(np.mean([len(o.split())/max(len(s.split()),1)
                           for o, s in zip(outs, EV_SRC)]))
    m = {"semantic_similarity": round(sim, 4),
         "lexical_diversity": round(100 - b_in, 2),   # 100 = nothing copied
         "bleu_vs_input": round(b_in, 2),
         "bleu_vs_reference": round(b_ref, 2),
         "ibleu": round(ibleu, 2),
         "rougeL_vs_reference": round(rl, 2),
         "copy_rate_pct": round(copy, 2),
         "length_ratio": round(lenr, 3)}
    print(f"{name:14} sim {m['semantic_similarity']:.3f} | diversity "
          f"{m['lexical_diversity']:.1f} | iBLEU {m['ibleu']:.2f} | copy {m['copy_rate_pct']:.1f}%")
    return m

@torch.no_grad()
def generate(mdl, srcs, tag, batch_size=16):
    FastModel.for_inference(mdl)
    side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    outs, t0 = [], time.time()
    for i in range(0, len(srcs), batch_size):
        chunk = srcs[i:i+batch_size]
        prompts = [render(s, generation=True) for s in chunk]
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        add_special_tokens=False).to("cuda")
        gen = mdl.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, use_cache=True,
                           pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        for j in range(len(chunk)):
            txt = tokenizer.decode(gen[j][enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True).strip()
            outs.append(txt.split("\n")[0].strip().strip('"') or chunk[j])
        print(f"  {tag} {min(i+batch_size,len(srcs))}/{len(srcs)} "
              f"({time.time()-t0:.0f}s)", end="\r")
    tokenizer.padding_side = side
    print(" " * 50, end="\r")
    return outs

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 7. The two baselines

In [8]:
clear()
COPY_M = evaluate_system(list(EV_SRC), "copy input")
BASE_OUTS = generate(model, EV_SRC, "base")
BASE_M = evaluate_system(BASE_OUTS, "base 0-shot")
print("\nsample base output:\n  src:", EV_SRC[0], "\n  out:", BASE_OUTS[0])
clear()

Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


copy input     sim 1.000 | diversity -0.0 | iBLEU -1.24 | copy 100.0%


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


base 0-shot    sim 0.909 | diversity 24.6 | iBLEU 1.65 | copy 41.7%

sample base output:
  src: Where from can I get best UX design course in India? 
  out: Where can I get best UX design course in India?


## 8. Training data and trainer

**The learning rate is the thing people get wrong here.** LoRA runs at 1e-4 to 5e-4
because a handful of parameters carry the whole update. Full fine-tuning moves every
weight, so the same rate destroys the pretrained model — it reads as the model "forgetting
how to speak". `1e-5` is the full fine-tuning range, ten to twenty times lower.

In [9]:
texts = [render(s, t) for s, t in TRAIN_PAIRS]
ds = Dataset.from_dict({"text": texts})

lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in texts[:2000]]
print(f"token length: median {int(np.median(lens))}, 95th {int(np.percentile(lens,95))}, "
      f"max {max(lens)} (cap {MAX_SEQ})")

try:
    from trl import SFTConfig
    CFG_CLS = SFTConfig
except ImportError:
    from transformers import TrainingArguments as CFG_CLS
_P = set(inspect.signature(CFG_CLS.__init__).parameters)

wanted = dict(
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
    warmup_ratio=0.03, warmup_steps=36, max_steps=MAX_STEPS, learning_rate=LR,
    fp16=False, bf16=BF16, logging_steps=10, optim="adamw_8bit",   # never fp16 for Gemma 3
    weight_decay=0.01, lr_scheduler_type="cosine", max_grad_norm=1.0,
    seed=SEED, output_dir="/content/outputs", report_to="none",
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ, dataset_text_field="text", dataset_num_proc=2, packing=False,
)
args_kw = {k: v for k, v in wanted.items() if k in _P}
if "warmup_ratio" in args_kw and "warmup_steps" in args_kw:
    args_kw.pop("warmup_steps")
dropped = sorted(set(wanted) - set(args_kw))
trainer_kw = {k: wanted[k] for k in
              ["max_seq_length","dataset_text_field","dataset_num_proc","packing"]
              if k in dropped}
if dropped:
    print("moved out of the config class for this version:", dropped)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
                     args=CFG_CLS(**args_kw), **trainer_kw)
# Gemma turn markers, not Llama's header tokens. Unsloth can auto-detect for Gemma 3,
# so try that first and fall back to explicit markers.
try:
    trainer = train_on_responses_only(trainer)
except Exception:
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<start_of_turn>user\n",
        response_part="<start_of_turn>model\n")

lab = np.array(trainer.train_dataset[0]["labels"])
sup = int((lab != -100).sum())
print(f"\n{len(lab)} tokens, {sup} supervised ({sup/len(lab):.1%})")
print("supervised:", repr(tokenizer.decode([t for t in lab if t != -100])[:120]))
assert 0 < sup < len(lab), "Response-only masking is not working."

token length: median 68, 95th 115, max 173 (cap 256)
moved out of the config class for this version: ['warmup_ratio']
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/40000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Unsloth: Auto-detected instruction_part = '<start_of_turn>user\n' and response_part = '<start_of_turn>model\n'


Map (num_proc=2):   0%|          | 0/40000 [00:00<?, ? examples/s]


82 tokens, 22 supervised (26.8%)
supervised: 'Can the barcode obtained through the IMEI method on my Samsung phone activate my Jio SIM despite any potential issues?<e'


## 9. Train

In [10]:
clear()
t0 = time.time()
stats = trainer.train()
TRAIN_MIN = (time.time()-t0)/60
VRAM_TRAIN = vram()
LOSS_CURVE = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
print(f"\n{TRAIN_MIN:.1f} min | peak VRAM {VRAM_TRAIN:.2f} GB of 15.0 | "
      f"final loss {stats.training_loss:.4f}")
assert not np.isnan(stats.training_loss), "Loss is NaN — lower the learning rate."

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40,000 | Num Epochs = 1 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 697,896,064 of 999,885,952 (69.80% trained)


Step,Training Loss
10,1.573800
20,0.837267
30,0.841565
40,0.812165
50,0.812134
60,0.875925
70,0.727647
80,0.850824
90,0.787169
100,0.796260


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/outputs/checkpoint-500.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/outputs/checkpoint-2000.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/outputs/checkpoint-2500/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/outputs/checkpoint-2500.



155.0 min | peak VRAM 9.44 GB of 15.0 | final loss 0.6825


## 10. Evaluate

In [11]:
clear()
TUNED_OUTS = generate(model, EV_SRC, "tuned")
TUNED_M = evaluate_system(TUNED_OUTS, "fine-tuned")

print(f"\n{'':22}{'copy':>9}{'base':>9}{'tuned':>9}")
for k in ["semantic_similarity","lexical_diversity","ibleu","rougeL_vs_reference",
          "copy_rate_pct","length_ratio"]:
    print(f"{k:22}{COPY_M[k]:>9}{BASE_M[k]:>9}{TUNED_M[k]:>9}")

D_IBLEU = TUNED_M["ibleu"] - BASE_M["ibleu"]
print(f"\niBLEU vs base: {D_IBLEU:+.2f}")

EXAMPLES = [{"source": EV_SRC[i], "reference": EV_REF[i],
             "base": BASE_OUTS[i], "tuned": TUNED_OUTS[i]} for i in range(5)]
for e in EXAMPLES[:3]:
    print(f"\nSRC   {e['source']}\nREF   {e['reference']}\nBASE  {e['base']}\nTUNED {e['tuned']}")

Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


fine-tuned     sim 0.911 | diversity 54.9 | iBLEU 17.99 | copy 0.7%

                           copy     base    tuned
semantic_similarity         1.0   0.9095   0.9114
lexical_diversity          -0.0     24.6     54.9
ibleu                     -1.24     1.65    17.99
rougeL_vs_reference       45.38    41.87    59.06
copy_rate_pct             100.0    41.67     0.67
length_ratio                1.0    1.053    1.095

iBLEU vs base: +16.34

SRC   Where from can I get best UX design course in India?
REF   What are the top-rated UX design courses available in India?
BASE  Where can I get best UX design course in India?
TUNED What is the best place to get UX design course in India?

SRC   Are there any Progressive Christian dating websites?
REF   Is there a dating website for Progressive Christians?
BASE  Are there any dating websites for Christian dating?
TUNED Are there any websites that cater to Progressive Christian dating?

SRC   Why is my dog shaking so much?
REF   What is causing my 

## 11. Charts

In [ ]:
# 1. the tradeoff plot — the chart that makes the task legible
fig, ax = plt.subplots(figsize=(5.8, 4.4))
pts = [("copy the input", COPY_M, C_COPY, "o"),
       ("base, zero-shot", BASE_M, C_BASE, "s"),
       ("fine-tuned", TUNED_M, C_TUNED, "D")]
ax.axhspan(0.80, 1.02, xmin=0.45, color=C_TUNED, alpha=.06)
ax.text(99, 0.815, "useful region", fontsize=8.5, color="#0f766e", ha="right")
for lab, m, c, mk in pts:
    ax.scatter(m["lexical_diversity"], m["semantic_similarity"], s=170, color=c,
               marker=mk, zorder=3, edgecolor="white", linewidth=1.4)
    ax.annotate(lab, (m["lexical_diversity"], m["semantic_similarity"]),
                textcoords="offset points", xytext=(0, -20), ha="center", fontsize=9)
ax.set_xlabel("lexical diversity  (100 − BLEU against the input)")
ax.set_ylabel("semantic similarity to the input")
ax.set_xlim(-4, 104); ax.set_ylim(0.55, 1.04)
ax.set_title("Paraphrasing is a trade-off, not a score", fontsize=11.5, pad=12)
plt.savefig(f"{ASSETS}/tradeoff.png"); plt.close()

# 2. metric bars
fig, ax = plt.subplots(figsize=(7.0, 3.5))
keys = [("ibleu","iBLEU"), ("bleu_vs_reference","BLEU vs ref"),
        ("rougeL_vs_reference","ROUGE-L"), ("copy_rate_pct","copy rate %")]
x = np.arange(len(keys)); w = 0.26
for off, (lab, m, c) in zip([-w, 0, w],
        [("copy", COPY_M, C_COPY), ("base", BASE_M, C_BASE), ("tuned", TUNED_M, C_TUNED)]):
    vals = [m[k] for k, _ in keys]
    ax.bar(x+off, vals, w, label=lab, color=c)
    for xi, v in zip(x+off, vals):
        ax.text(xi, v+.8, f"{v:.1f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([n for _, n in keys])
ax.legend(frameon=False, fontsize=9, ncol=3)
ax.set_title(f"{EVAL_N} held-out pairs, greedy decoding", fontsize=11, pad=10)
plt.savefig(f"{ASSETS}/metrics.png"); plt.close()

# 3. loss
fig, ax = plt.subplots(figsize=(6.0, 3.2))
ax.plot([s for s,_ in LOSS_CURVE], [l for _,l in LOSS_CURVE], color=C_TUNED, lw=1.5)
ax.set_xlabel("step"); ax.set_ylabel("training loss")
ax.set_title("Training loss (assistant turn only)", fontsize=11, pad=10)
plt.savefig(f"{ASSETS}/loss-curve.png"); plt.close()

# 4. memory
fig, ax = plt.subplots(figsize=(5.4, 3.3))
est_lora = VRAM_LOAD + 1.2
vals = [VRAM_LOAD, VRAM_TRAIN, est_lora]
bars = ax.bar(["weights\nafter load", "peak during\nfull FT", "LoRA would use\n(estimate)"],
              vals, color=[C_BASE, C_TUNED, C_COPY], width=.55)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+.2, f"{v:.1f}", ha="center", fontsize=9)
ax.axhline(15.0, color="black", ls="--", lw=1)
ax.text(2.45, 15.3, "T4 limit", fontsize=8, ha="right")
ax.set_ylabel("GB"); ax.set_ylim(0, 17)
ax.set_title("Full fine-tuning costs what it costs", fontsize=11, pad=10)
plt.savefig(f"{ASSETS}/memory.png"); plt.close()

# 5. length ratio
fig, ax = plt.subplots(figsize=(6.0, 3.2))
for outs, lab, c in [(BASE_OUTS,"base",C_BASE), (TUNED_OUTS,"fine-tuned",C_TUNED)]:
    r = [len(o.split())/max(len(s.split()),1) for o, s in zip(outs, EV_SRC)]
    ax.hist(r, bins=np.linspace(0, 3, 31), alpha=.62, label=lab, color=c)
ax.axvline(1.0, color="black", ls="--", lw=1)
ax.text(1.03, ax.get_ylim()[1]*.92, "same length as input", fontsize=8)
ax.set_xlabel("output length ÷ input length"); ax.set_ylabel("paraphrases")
ax.legend(frameon=False, fontsize=9)
ax.set_title("Does it rewrite, or does it pad?", fontsize=11, pad=10)
plt.savefig(f"{ASSETS}/length.png"); plt.close()

print("charts:", sorted(os.listdir(ASSETS)))
from IPython.display import Image, display
for f in ["tradeoff.png", "metrics.png"]:
    display(Image(f"{ASSETS}/{f}"))

## 12. Save and log in

In [13]:
model.save_pretrained_merged(SAVE_DIR, tokenizer, save_method="merged_16bit")
size_gb = sum(os.path.getsize(os.path.join(SAVE_DIR, f))
              for f in os.listdir(SAVE_DIR)
              if os.path.isfile(os.path.join(SAVE_DIR, f))) / 1e9
print(f"model saved: {size_gb:.2f} GB")

with open(f"{SAVE_DIR}/prompt_format.json", "w") as f:
    json.dump({"chat_template":"gemma-3","system_role_supported":False,
               "user_instruction":INSTRUCTION,
               "note":"Gemma's template emits <bos>; strip the tokenizer's duplicate.",
               "rendered_example": render("The cat sat on the mat.", generation=True)},
              f, indent=2, ensure_ascii=False)

KWH = 0.07 * (TRAIN_MIN/60)        # T4 draws ~70 W under load
CO2 = KWH * 0.417                  # global average grid, kg CO2e per kWh

from huggingface_hub import notebook_login, create_repo, upload_folder
notebook_login()

Unsloth: Saving full fine-tuned model to '/content/gemma-3-1b-paraphrase' ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/gemma-3-1b-paraphrase/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/gemma-3-1b-paraphrase.


Unsloth: Model saved successfully to '/content/gemma-3-1b-paraphrase'
model saved: 2.64 GB


### EDIT THIS

In [ ]:
HF_USERNAME = "emonHasan01"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"
print("publishing to:", REPO_ID)

## 13. Model card

In [ ]:
def c(s, n=62):
    s = " ".join(str(s).split())[:n].rstrip("\\")
    return s.replace("|", "\\|")

ex_rows = "\n".join(f"| {c(e['source'])} | {c(e['reference'])} | {c(e['base'])} | "
                    f"{c(e['tuned'])} |" for e in EXAMPLES)

def row(m):
    return (f"| {m['semantic_similarity']:.3f} | {m['lexical_diversity']:.1f} | "
            f"{m['ibleu']:.2f} | {m['bleu_vs_reference']:.2f} | "
            f"{m['rougeL_vs_reference']:.2f} | {m['copy_rate_pct']:.1f}% | "
            f"{m['length_ratio']:.2f} |")

if D_IBLEU > 2:
    headline = (f"iBLEU rises from {BASE_M['ibleu']:.2f} to {TUNED_M['ibleu']:.2f}, with "
                f"semantic similarity held at {TUNED_M['semantic_similarity']:.3f}.")
elif D_IBLEU > -2:
    headline = (f"iBLEU is close to the base model's ({BASE_M['ibleu']:.2f} to "
                f"{TUNED_M['ibleu']:.2f}). The clearer change is in format discipline — "
                "the fine-tuned model returns a bare paraphrase instead of a preamble.")
else:
    headline = (f"iBLEU **falls** from {BASE_M['ibleu']:.2f} to {TUNED_M['ibleu']:.2f}. "
                "Published as a measured result; see the trade-off plot before drawing "
                "conclusions from a single number.")

CARD = """---
language:
  - en
license: gemma
library_name: transformers
pipeline_tag: text2text-generation
base_model: __BASE__
base_model_relation: finetune
tags:
  - paraphrase
  - paraphrase-generation
  - text-rewriting
  - full-finetune
  - unsloth
  - gemma3
datasets:
  - __DATASET__
metrics:
  - bleu
  - rouge
model-index:
  - name: __MODEL_NAME__
    results:
      - task:
          type: text2text-generation
          name: Paraphrase generation
        dataset:
          name: __DATASET__
          type: __DATASET__
          split: held-out
        metrics:
          - type: bleu
            name: iBLEU (__EVAL_N__ pairs)
            value: __IBLEU__
          - type: rouge
            name: ROUGE-L vs reference
            value: __ROUGEL__
---

<img src="assets/banner.png" alt="__MODEL_NAME__" width="100%">

# __MODEL_NAME__

**A 1B paraphraser, fully fine-tuned.** Every weight of
[`__BASE__`](https://huggingface.co/__BASE__) was updated — no adapters — using
[Unsloth](https://github.com/unslothai/unsloth) on
[`__DATASET__`](https://huggingface.co/datasets/__DATASET__), on a **free Colab T4** in
__MINS__ minutes.

__HEADLINE__

## Why one number cannot grade a paraphraser

A paraphrase has to do two things that pull against each other:

- **keep the meaning** — maximised by returning the input unchanged
- **change the wording** — maximised by writing something unrelated

So any single metric can be gamed. Copying the input scores a perfect **1.000** semantic
similarity and is worthless. This card therefore reports both axes, and includes a **copy
baseline** so the trade-off is visible rather than implied.

![Trade-off](assets/tradeoff.png)

Read that plot before the tables. Top-right is good; anything near the top-left corner is
a model that has learned to echo.

## Results

__EVAL_N__ held-out pairs, greedy decoding, identical prompt for every system.

| | semantic similarity ↑ | lexical diversity ↑ | iBLEU ↑ | BLEU vs ref ↑ | ROUGE-L ↑ | copy rate ↓ | length ratio |
|---|---|---|---|---|---|---|---|
| copy the input __COPY_ROW__
| `__BASE__` zero-shot __BASE_ROW__
| **this model** __TUNED_ROW__

![Metrics](assets/metrics.png)

**How to read each column.** *Semantic similarity* is cosine distance between
`all-MiniLM-L6-v2` embeddings of input and output. *Lexical diversity* is
`100 − BLEU(output, input)`, so higher means less copying. *iBLEU* is the standard combined
paraphrase measure, `0.8 × BLEU(output, reference) − 0.2 × BLEU(output, input)` — it rewards
matching a human paraphrase and penalises echoing the source. *Copy rate* is the share of
outputs identical to the input.

![Length ratio](assets/length.png)

![Training loss](assets/loss-curve.png)

## Examples

| Source | Human reference | Base, zero-shot | This model |
|---|---|---|---|
__EX_ROWS__

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("__REPO_ID__")
model = AutoModelForCausalLM.from_pretrained("__REPO_ID__", torch_dtype=torch.float16,
                                             device_map="auto")

# Gemma 3 has no system role — the instruction goes in the user turn
INSTRUCTION = ("Rewrite the sentence below. Keep the meaning exactly the same, change the "
               "wording and structure, and return only the rewritten sentence.\\n\\n{src}")

messages = [{"role": "user", "content": INSTRUCTION.format(src="The cat sat on the mat.")}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt = prompt.removeprefix("<bos>")   # the tokenizer adds one; two degrades output
inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
out = model.generate(**inputs, max_new_tokens=72, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

**Several different paraphrases of the same sentence:**

```python
out = model.generate(**inputs, max_new_tokens=72, do_sample=True,
                     temperature=0.9, top_p=0.95, num_return_sequences=4)
```

Greedy decoding reproduces the numbers above. Sampling at `temperature=0.9` gives variety
at some cost to fidelity — raise it for brainstorming, keep it greedy when the meaning must
hold. Google's recommended settings for general Gemma 3 use are `temperature=1.0`,
`top_k=64`, `top_p=0.95`; for paraphrasing those produce noticeably looser rewrites.

**The instruction wording is part of the model.** Gemma 3's chat template has no `system`
role, so the instruction was folded into the user turn — exactly as shown above — in every
training example. A different wrapper degrades the output format. Exact strings in
[`prompt_format.json`](prompt_format.json).

**Strip the duplicate `<bos>`.** Gemma's chat template emits one, and the tokenizer adds
another unless you remove it. Two BOS tokens measurably degrade quality, and nothing warns
you.

## Training

| | |
|---|---|
| Method | **Full fine-tuning** — every non-embedding weight updated |
| Base model | `__BASE__` (__PARAMSB__B parameters), loaded from the `unsloth/` mirror |
| Trainable | __TRAINABLE__ (__TRAINPCT__%); embeddings frozen (__FROZENPCT__%) |
| Learning rate | __LR__, cosine schedule, 3% warmup |
| Batch | __BATCH__ × __ACCUM__ accumulation = __EFF__ effective |
| Steps | __STEPS__ (~__SEEN__ examples) |
| Max sequence | __MAXSEQ__ tokens |
| Optimizer | `adamw_8bit`, `max_grad_norm=1.0`, gradient checkpointing |
| Precision | **pure float32** — Gemma 3 overflows float16 (see below) |
| Peak VRAM | __VRAMTRAIN__ GB of 15.0 |
| Hardware | __GPU__, free Colab tier |
| Training time | ~__MINS__ minutes |
| Energy | ~__KWH__ kWh, ≈__CO2__ kg CO₂e at the global average grid |

![Memory](assets/memory.png)

### Decisions that mattered

**Learning rate `__LR__`, not `2e-4`.** This is the most common way to ruin a full
fine-tune. LoRA uses 1e-4 to 5e-4 because a few million parameters carry the entire update.
Full fine-tuning moves every weight, so that rate overwrites the pretraining and the model
stops writing fluent English. Full fine-tuning lives at 1e-5 to 2e-5.

**Embeddings frozen.** Gemma 3's vocabulary is 262,144 tokens, so `embed_tokens` is
262,144 × 1,152 = 302M parameters — __FROZENPCT__% of this entire model, with `lm_head` tied
to it. Paraphrasing adds no vocabulary, so training it buys very little while costing
gradients plus optimizer state for nearly a third of the network. In fp32 that is the
difference between fitting on a T4 and not: peak memory lands at __VRAMTRAIN__ GB rather
than against the 15 GB ceiling.

**Gemma 3 cannot be trained in float16 at all.** Its activations exceed float16's maximum
of 65,504, so gradients and activations become infinity on any card with float16 tensor
cores but no bfloat16 — T4, V100, RTX 20-series. This is the model's dynamic range, not a
bug in the training script, and Unsloth's remedy is to default every layer to float32.
Training here therefore runs in **pure fp32**: slower than mixed precision, and every
memory estimate that assumes 2 bytes per weight is wrong by half. On Ampere or newer,
bfloat16 sidesteps the whole problem.

**`FastModel`, not `FastLanguageModel`.** Gemma 3 needs the multimodal-aware loader even
for the text-only 1B, and the older loader fails or misconfigures the model.

**Loss on the assistant turn only.** `train_on_responses_only` masks the prompt. Otherwise
most of the gradient goes into regenerating the input sentence, which is always supplied.

**Pairs filtered and deduplicated.** Identical pairs, length ratios outside 0.5–2.0, and
repeated source sentences were dropped before training. Duplicate sources inflate held-out
scores by leaking training text into evaluation.

### Would LoRA have done as well?

Probably. Unsloth's own guidance is that full fine-tuning is usually unnecessary and that
LoRA matches it when done properly. This model is a full fine-tune because that was the
brief, and because the memory wall is worth seeing. **It is not evidence that full
fine-tuning beats LoRA here** — that comparison was not run, and the card would say so if
it had been.

## Intended use

Rewriting a sentence while preserving meaning: varying phrasing in drafts, generating
training data, simplifying repetitive text, producing alternatives for a human to choose
between.

## Out of scope

- **Evading plagiarism or AI detection.** Rewording a source does not make it yours, and
  the model was not trained or evaluated for that.
- **Legal, medical or safety-critical text**, where a subtle meaning shift matters and
  nothing here verifies that meaning survived.
- **Anything non-English.** Gemma 3 is multilingual, but this fine-tune used English data
  only and was evaluated in English only. Other languages are unmeasured and the instruction
  format was never trained in them.

## Limitations

- **Semantic similarity is measured by a 22M-parameter embedding model**, not by a human.
  It can rate a meaning-changing rewrite as similar, particularly when a negation flips or
  a number changes. No human evaluation was run.
- **One paraphrase per input during evaluation.** Diversity across samples was not measured.
- **Evaluated on __EVAL_N__ pairs** from one dataset. Expect sampling noise of a point or
  two on BLEU-based numbers.
- **The references are themselves machine-generated** (ChatGPT paraphrases), so BLEU against
  them measures agreement with another model, not with a human standard.
- **Sentence-length inputs.** Trained at __MAXSEQ__ tokens; paragraphs and documents are out
  of distribution.
- **It can drop or alter details** — numbers, names, negations — while producing fluent,
  confident output. Check anything that matters.
- **1B parameters.** Dedicated paraphrase models and larger general models will do better.
  This is a small model trained on a free GPU in under two hours.

## Reproducing

One Colab notebook on a __GPU__ produced the copy baseline, the zero-shot baseline, the
training, all five charts and this card. Raw numbers in
[`results.json`](results.json), prompt in [`prompt_format.json`](prompt_format.json).

## Citation

```bibtex
@software{unsloth,
  title  = {Unsloth},
  author = {Daniel Han and Michael Han and Unsloth team},
  year   = {2023},
  url    = {https://github.com/unslothai/unsloth}
}
@inproceedings{sun2012ibleu,
  title     = {Joint Learning of a Dual SMT System for Paraphrase Generation},
  author    = {Sun, Hong and Zhou, Ming},
  booktitle = {ACL},
  year      = {2012}
}
```
"""

repl = {
    "__BASE__": UPSTREAM, "__MODEL_NAME__": MODEL_NAME, "__REPO_ID__": REPO_ID,
    "__DATASET__": DATASET, "__UPSTREAM__": UPSTREAM,
    "__HEADLINE__": headline, "__EVAL_N__": str(EVAL_N),
    "__COPY_ROW__": row(COPY_M), "__BASE_ROW__": row(BASE_M), "__TUNED_ROW__": row(TUNED_M),
    "__IBLEU__": f"{TUNED_M['ibleu']:.2f}", "__ROUGEL__": f"{TUNED_M['rougeL_vs_reference']:.2f}",
    "__EX_ROWS__": ex_rows,
    "__PARAMSB__": f"{N_PARAMS/1e9:.2f}", "__TRAINABLE__": f"{TRAINABLE:,}",
    "__TRAINPCT__": f"{100*TRAINABLE/N_PARAMS:.1f}",
    "__FROZENPCT__": f"{100*FROZEN/N_PARAMS:.1f}",
    "__LR__": f"{LR:g}", "__BATCH__": str(BATCH), "__ACCUM__": str(ACCUM),
    "__EFF__": str(BATCH*ACCUM), "__STEPS__": str(MAX_STEPS),
    "__SEEN__": f"{MAX_STEPS*BATCH*ACCUM:,}", "__MAXSEQ__": str(MAX_SEQ),
    "__VRAMTRAIN__": f"{VRAM_TRAIN:.1f}", "__GPU__": GPU,
    "__MINS__": f"{TRAIN_MIN:.0f}", "__KWH__": f"{KWH:.2f}", "__CO2__": f"{CO2:.3f}",
}
for k, v in repl.items():
    CARD = CARD.replace(k, v)
leftover = sorted(set(re.findall(r"__[A-Z0-9_]+__", CARD)))
assert not leftover, f"Unresolved placeholders: {leftover}"
assert CARD.startswith("---\n"), "YAML front matter malformed"

if not os.path.exists(f"{ASSETS}/banner.png"):
    CARD = re.sub(r'<img src="assets/banner\.png".*?>\n\n', "", CARD, flags=re.S)
    print("no banner.png in assets/ — <img> line removed")

with open(f"{SAVE_DIR}/README.md", "w", encoding="utf-8") as f:
    f.write(CARD)

RESULTS = {
    "base_model": BASE_MODEL, "dataset": DATASET, "eval_pairs": EVAL_N,
    "metrics": {"copy_input": COPY_M, "base_zero_shot": BASE_M, "fine_tuned": TUNED_M},
    "delta_ibleu_vs_base": D_IBLEU, "loss_curve": LOSS_CURVE, "examples": EXAMPLES,
    "config": {"method":"full fine-tuning (embeddings frozen)", "lr":LR,
               "batch":BATCH, "accum":ACCUM, "steps":MAX_STEPS, "max_seq":MAX_SEQ,
               "optim":"adamw_8bit", "total_params":N_PARAMS, "trainable_params":TRAINABLE,
               "frozen_params":FROZEN, "train_examples_seen":MAX_STEPS*BATCH*ACCUM,
               "train_pairs":len(TRAIN_PAIRS), "vram_load_gb":round(VRAM_LOAD,2),
               "vram_peak_gb":round(VRAM_TRAIN,2), "train_minutes":round(TRAIN_MIN,1),
               "kwh":round(KWH,3), "co2e_kg":round(CO2,4),
               "hardware":GPU, "seed":SEED, "model_gb":round(size_gb,2)},
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False)
print("card written\n"); print(CARD[:1100])

## 14. Push

In [16]:
create_repo(REPO_ID, repo_type="model", private=False, exist_ok=True)
upload_folder(repo_id=REPO_ID, folder_path=SAVE_DIR,
              commit_message="Full fine-tune for paraphrase generation, with copy and zero-shot baselines")
print(f"https://huggingface.co/{REPO_ID}")

https://huggingface.co/emonHasan01/gemma-3-1b-paraphrase


## 15. Verify from a clean load

In [17]:
del model, trainer; clear()
from transformers import AutoTokenizer as AT, AutoModelForCausalLM as AM

tok2 = AT.from_pretrained(REPO_ID)
m2 = AM.from_pretrained(REPO_ID, torch_dtype=torch.float16, device_map="auto").eval()

for s in ["The committee approved the proposal after a lengthy debate.",
          "She finished the marathon despite injuring her ankle at mile eight."]:
    p = tok2.apply_chat_template(to_chat(s), tokenize=False,
                                 add_generation_prompt=True).removeprefix("<bos>")
    enc = tok2(p, return_tensors="pt", add_special_tokens=False).to(m2.device)
    with torch.no_grad():
        o = m2.generate(**enc, max_new_tokens=72, do_sample=False)
    print(f"IN   {s}\nOUT  {tok2.decode(o[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()}\n")

config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=72) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


IN   The committee approved the proposal after a lengthy debate.
OUT  The committee approved the proposal following a lengthy debate.

IN   She finished the marathon despite injuring her ankle at mile eight.
OUT  Despite injuring her ankle at mile eight, she completed the marathon.



In [1]:
!pip install -q "transformers>=4.49,<5" optimum-onnx "optimum>=2.0" onnx onnxruntime huggingface_hub sentencepiece
!pip uninstall -y -q torchao diffusers
print("installed — if this is a fresh runtime you can continue straight on")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have hugging

In [2]:
import os, re, json, shutil, subprocess, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import notebook_login, create_repo, upload_folder

MODEL_REPO = "emonHasan01/gemma-3-1b-paraphrase"
ONNX_REPO  = "emonHasan01/gemma-3-1b-paraphrase-ONNX"
SPACE_REPO = "emonHasan01/say-it-another-way"

# The instruction the model was fine-tuned with. If your training prompt differed,
# change this one line — the Space uses it verbatim.
INSTRUCTION = "Paraphrase the following sentence:\n\n{text}"

EXPORT = "/content/onnx_export"
STAGE  = "/content/web_build"
SPACE  = "/content/space"
for d in (STAGE, SPACE):
    os.makedirs(d, exist_ok=True)

notebook_login()

In [3]:
PROBES = [
    "The committee decided to postpone the meeting until further notice.",
    "She walked quickly down the street because she was late for work.",
]

tok = AutoTokenizer.from_pretrained(MODEL_REPO)
torch_model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, dtype=torch.float32)
torch_model.eval()

def paraphrase(model, tokenizer, text, max_new_tokens=96):
    msgs = [{"role": "user", "content": INSTRUCTION.format(text=text)}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

TORCH_REF = [paraphrase(torch_model, tok, p) for p in PROBES]
for p, r in zip(PROBES, TORCH_REF):
    print(f"IN   {p}\nOUT  {r}\n")

del torch_model
import gc; gc.collect()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.60G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


IN   The committee decided to postpone the meeting until further notice.
OUT  The meeting was postponed until further notice, as determined by the committee.

IN   She walked quickly down the street because she was late for work.
OUT  She hurried down the street due to her tardiness for work.



89